## Computational Physics -- Problem Set 3
##### _Theodore Landa_

In [ ]:
import numpy as np
import math
import time
import scipy.integrate as sp
from functools import reduce
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

### Problem 1.
np.sum() agrees with my three other methods: we get `1.5707946637152908` for `np.sum()`, `1.5707946637152863` for the `for` loop, `1.5707946637152863` for the while loop, and `1.5707946637152863` for `map reduce` and `reduce`. The most natural of these seems to be `np.sum()`, which we can justify by recalling that `np.sum()` employs a whole lot of fast backend and parallelism. We can also justify this by looking at my figures (both linear and loglog) which indicate that `np.sum()` destroys the other methods. Not counting `np.sum()` though, `reduce` is the best option (using numpy for the map operation, rather than python's built in map, which is slower). This is probably because reduce has some kind of quicker iteration/indexing scheme going on.

In [ ]:
def f2(x):
    return np.sqrt(1-x**2)

In [ ]:
def points(N, start, stop):
    x = np.linspace(start, stop, N, endpoint=False)
    return x, (stop-start)/N

In [ ]:
def npsum(f, x, h):
    starttime = time.time()
    y = f(x)
    I = np.sum(y)*h
    endtime = time.time()
    cost = endtime - starttime
    return I, cost

In [ ]:
def forsum(f, x, h):
    starttime = time.time()
    y = f(x)
    I = 0
    for j in range(np.size(x)):
        I += y[j]
    I = I*h
    endtime = time.time()
    cost = endtime - starttime
    return I, cost

In [ ]:
def whilesum(f, x, h):
    starttime = time.time()
    y = f(x)
    I = 0
    j = 0
    while (j < np.size(x)):
        I += y[j]
        j += 1
    I = I*h
    endtime = time.time()
    cost = endtime - starttime
    return I, cost

In [ ]:
def redsum(f, x, h):
    starttime = time.time()
    y = f(x)
    I = reduce(lambda u, v: u + v, y) * h
    endtime = time.time()
    cost = endtime - starttime
    return I, cost

In [ ]:
def mapredsum(f, x, h):
    starttime = time.time()
    y = map(lambda u: f(u), x)
    I = reduce(lambda u, v: u + v, y) * h
    endtime = time.time()
    cost = endtime - starttime
    return I, cost

In [ ]:
x, h = points(int(1e4), -1, 1)
print(npsum(f2, x, h)[0])
print(forsum(f2, x, h)[0])
print(whilesum(f2, x, h)[0])
print(redsum(f2, x, h)[0])
print(mapredsum(f2, x, h)[0])

In [ ]:
numNs = 1000
Ns = np.logspace(2, 6, numNs, dtype=np.int32)
nptimes = np.zeros(numNs)
fortimes = np.zeros(numNs)
whiletimes = np.zeros(numNs)
redtimes = np.zeros(numNs)
mapredtimes = np.zeros(numNs)
for j in range(numNs):
    x, h = points(Ns[j], -1, 1)
    nptimes[j] = npsum(f2, x, h)[1]
    fortimes[j] = forsum(f2, x, h)[1]
    whiletimes[j] = whilesum(f2, x, h)[1]
    redtimes[j] = redsum(f2, x, h)[1]
    mapredtimes[j] = mapredsum(f2, x, h)[1]

In [ ]:
plt.plot(Ns, nptimes, 'k.', label='np.sum()', markersize=3)
plt.plot(Ns, fortimes, 'g.', label='for', markersize=3)
plt.plot(Ns, whiletimes, 'r.', label='while', markersize=3)
plt.plot(Ns, redtimes, 'm.', label='reduce', markersize=3)
plt.plot(Ns, mapredtimes, 'b.', label='map reduce', markersize=3)
plt.xlabel(r'Number of evals $N$')
plt.ylabel(r'Wall Clock Time $T$')
plt.title(r'How Long is this taking?')
plt.legend()

In [ ]:
plt.loglog(Ns, nptimes, 'k.', label='np.sum()', markersize=3)
plt.loglog(Ns, fortimes, 'g.', label='for', markersize=3)
plt.loglog(Ns, whiletimes, 'r.', label='while', markersize=3)
plt.loglog(Ns, redtimes, 'm.', label='reduce', markersize=3)
plt.loglog(Ns, mapredtimes, 'b.', label='map reduce', markersize=3)
plt.xlabel(r'Number of evals $N$')
plt.ylabel(r'Wall Clock Time $T$')
plt.title(r'How Long is this taking?')
plt.legend()

### Problem 2.
_(a)_ We have written the program and computed the value `4.400426666666667` with 10 slices, which is off from the real value by `0.009696969696972665`%. For 100 slices and 1000 slices we get `4.400000042666667` and `4.400000000004267`, which are off from the real value by `9.69696969186564e-07`% and `9.697293473271367e-11`% respectively. These are huge improvements from 2%, 0.02%, and 0.0002%, which were the errors for the same respective numbers of slices using the trapezoidal rule.

_(b)_ We use `scipy.integrate.fixed_quad` and get `4.399999999999999`---accurate to machine precision---for 10, 100, and 1000 evaluations. That's wild.

_(c)_ Check out my plot!! Quadrature converges wayy faster, but Simpson's rule has a more 'regular' error!

In [ ]:
def f(x):
    return x**4 - 2*x + 1

In [ ]:
def simpsons(f, N, start, stop):
    if (N%2 == 1):
        print("Don't you DARE!!! Educate yourself. N should be even! Try again.")
    else:
        h = (stop-start)/N
        s1 = np.sum(f(start + ((2*np.linspace(1, N/2, int(N/2))) - 1)*h))
        s2 = np.sum(f(2*np.linspace(1, N/2 - 1, int(N/2) - 1)*h + start))
        return (1/3)*h*(f(start)+f(stop)+4*s1+2*s2)

In [ ]:
simp10 = simpsons(f, 10, 0., 2.)
simp100 = simpsons(f, 100, 0., 2.)
simp1000 = simpsons(f, 1000, 0., 2.)
print(simp10)
print(100*np.abs(simp10-4.4)/4.4)
print(simp100)
print(100*np.abs(simp100-4.4)/4.4)
print(simp1000)
print(100*np.abs(simp1000-4.4)/4.4)

In [ ]:
quad10 = sp.fixed_quad(f, 0., 2., n=10)
print(quad10)
quad100 = sp.fixed_quad(f, 0., 2., n=10)
print(quad10)
quad1000 = sp.fixed_quad(f, 0., 2., n=10)
print(quad10)

In [ ]:
numNs = 100
Ns = np.logspace(0, 3, numNs, dtype=np.int32)
Nseven = np.logspace(0, 2.7, numNs, dtype=np.int32)*2
mysimpsons = np.zeros(numNs)
#simpsons = np.zeros(numNs)
quads = np.zeros(numNs)
for j in range(numNs):
    mysimpsons[j] = np.abs(simpsons(f, Nseven[j], 0., 2.)-4.4)/4.4
    #simpsons[j] = sp.simpsons(f, j, 0., 2.)
    quads[j] = np.abs(sp.fixed_quad(f, 0., 2., n=Ns[j])[0]-4.4)/4.4

In [ ]:
plt.loglog(Ns, quads, label='quadrature')
plt.loglog(Nseven, mysimpsons, label="simpson's rule")
plt.xlabel(r'Number of Slices $N$')
plt.ylabel(r'Fractional Error')
plt.legend()

### Problem 3.
Check out my plot!

In [ ]:
def finfo(x):
    return f(x)+1e4+5000*np.sin(1.6*x-2)

fig, ax = plt.subplots(figsize=(7, 5))

c_euler = '#DDAA33'
c_trap = '#CC6677'
c_simp = '#4477AA'

xfunc = np.linspace(4, 14, 1000)
x1 = 8
x2 = 12
xmid = 10
y1 = finfo(x1)
y2 = finfo(x2)
ymid = finfo(xmid)
xarr = np.array([x1, xmid, x2])
yarr = np.array([y1, ymid, y2])
ax.plot(xfunc, finfo(xfunc), 'k')
ax.set_ylim(0, 4*1e4)
ax.set_xlim(7, 13)
xsecord = np.linspace(x1, x2, 500)
a, b, c = np.polyfit(xarr, yarr, 2)
m, b1 = np.polyfit([x1, x2], [y1, y2], 1)
ax.plot(xsecord, np.ones(500)*ymid, color=c_euler)
ax.fill_between(xsecord, 0, ymid, color=c_euler, alpha=0.25, label='Midpoint/Euler')
ax.plot(xsecord, m*xsecord+b1, color=c_trap)
ax.fill_between(xsecord, 0, m*xsecord+b1, color=c_trap, alpha=0.25, label='Trapezoidal')
def secord(x):
    return a*x**2+b*x+c
plt.plot(xsecord, secord(xsecord), color=c_simp)
ax.fill_between(xsecord, 0, secord(xsecord), color=c_simp, alpha=0.25, label="Simpson's")


ax.set_xticks([x1, xmid, x2])
ax.set_xticklabels([r'$x_0$', r'$x_0+h/2$', r'$x_0+h$'])
ax.set_yticks([])

ax.plot(xarr, yarr, 'k.', markersize = 7)
ax.plot(xfunc, finfo(xfunc), 'k')

ax.text(10, 16000, 'Trapezoidal', color=c_trap, fontsize=10)
ax.text(8.2, 23000, 'Euler/midpoint', color=c_euler, fontsize=10)
ax.text(10.7, 28000, "Simpson's", color=c_simp, fontsize=10)
ax.text(8.2, 8000, "f(x)", color='k', fontsize=10)

### Problem 4.
My error estimate on the 20 slice integral from integrating with 10 slices is `-0.026633333333333432`, whereas the true error is `-0.026659999999999684`. These are very close! But not the same. They aren't the same because we neglect higher order error terms in our estimate! We're technically doing an approximation that's second order in h, because the leading order term is quadratic in the step size.

In [ ]:
x10 = np.linspace(0., 2., 11)
y10 = f(x10)
x20 = np.linspace(0., 2., 21)
y20 = f(x20)
I10 = np.trapezoid(y10, x10)
I20 = np.trapezoid(y20, x20)
print((1/3)*(I20-I10))
print(4.4-I20)